In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import transformers
transformers.logging.set_verbosity_error()

In [ ]:
import os
os.makedirs("/content/drive/MyDrive/models", exist_ok=True)

In [ ]:
# Load and preprocess

import pandas as pd
import re
from sklearn.model_selection import train_test_split
import joblib

path = "/content/drive/MyDrive/colab_datasets/combined_hate_speech_dataset.csv"
df = pd.read_csv(path)

df = df[['text', 'hate_label']].dropna()

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    return text

df['clean_text'] = df['text'].apply(clean_text)

X_train, X_temp, y_train, y_temp = train_test_split(
    df['clean_text'], df['hate_label'],
    test_size=0.2, stratify=df['hate_label'], random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5, stratify=y_temp, random_state=42
)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1,2), stop_words='english')

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf   = tfidf.transform(X_val)
X_test_tfidf  = tfidf.transform(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(model, X, y):
    pred = model.predict(X)
    return {
        "accuracy": round(accuracy_score(y, pred), 4),
        "precision": round(precision_score(y, pred, average="weighted"), 4),
        "recall": round(recall_score(y, pred, average="weighted"), 4),
        "f1": round(f1_score(y, pred, average="weighted"), 4)
    }

In [ ]:
from sklearn.linear_model import LogisticRegression

model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X_train_tfidf, y_train)

print("\n=== Logistic Regression ===")
print("Train:", evaluate(model_lr, X_train_tfidf, y_train))
print("Val  :", evaluate(model_lr, X_val_tfidf, y_val))
print("Test :", evaluate(model_lr, X_test_tfidf, y_test))

joblib.dump(model_lr, "/content/drive/MyDrive/models/logistic_model.pkl")
joblib.dump(tfidf, "/content/drive/MyDrive/models/tfidf_vectorizer.pkl")


=== Logistic Regression ===
Train: {'accuracy': 0.7505, 'precision': 0.7797, 'recall': 0.7505, 'f1': 0.7392}
Val  : {'accuracy': 0.6782, 'precision': 0.698, 'recall': 0.6782, 'f1': 0.6619}
Test : {'accuracy': 0.6772, 'precision': 0.6968, 'recall': 0.6772, 'f1': 0.6607}


['/content/drive/MyDrive/models/tfidf_vectorizer.pkl']

In [ ]:
from sklearn.svm import LinearSVC

model_svm = LinearSVC()
model_svm.fit(X_train_tfidf, y_train)

print("\n=== SVM ===")
print("Train:", evaluate(model_svm, X_train_tfidf, y_train))
print("Val  :", evaluate(model_svm, X_val_tfidf, y_val))
print("Test :", evaluate(model_svm, X_test_tfidf, y_test))
joblib.dump(model_svm, "/content/drive/MyDrive/models/svm_model.pkl")


=== SVM ===
Train: {'accuracy': 0.8186, 'precision': 0.8484, 'recall': 0.8186, 'f1': 0.8121}
Val  : {'accuracy': 0.665, 'precision': 0.6756, 'recall': 0.665, 'f1': 0.6521}
Test : {'accuracy': 0.6677, 'precision': 0.68, 'recall': 0.6677, 'f1': 0.654}


['/content/drive/MyDrive/models/svm_model.pkl']

In [ ]:
import xgboost as xgb

model_xgb = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    tree_method="hist"
)

model_xgb.fit(X_train_tfidf, y_train)

print("\n=== XGBoost ===")
print("Train:", evaluate(model_xgb, X_train_tfidf, y_train))
print("Val  :", evaluate(model_xgb, X_val_tfidf, y_val))
print("Test :", evaluate(model_xgb, X_test_tfidf, y_test))
model_xgb.save_model("/content/drive/MyDrive/models/xgb_model.json")


=== XGBoost ===
Train: {'accuracy': 0.7177, 'precision': 0.7543, 'recall': 0.7177, 'f1': 0.7005}
Val  : {'accuracy': 0.6558, 'precision': 0.6815, 'recall': 0.6558, 'f1': 0.6321}
Test : {'accuracy': 0.6677, 'precision': 0.6975, 'recall': 0.6677, 'f1': 0.6444}


In [ ]:
!pip install transformers datasets -q

In [ ]:
def clean_eval(trainer, dataset):
    res = trainer.evaluate(dataset)
    return {
        "accuracy": round(res["eval_accuracy"], 4),
        "precision": round(res["eval_precision"], 4),
        "recall": round(res["eval_recall"], 4),
        "f1": round(res["eval_f1"], 4)
    }

In [ ]:
import transformers
transformers.logging.set_verbosity_error()

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import numpy as np

train_df = pd.DataFrame({'text': X_train, 'label': y_train})
val_df   = pd.DataFrame({'text': X_val, 'label': y_val})
test_df  = pd.DataFrame({'text': X_test, 'label': y_test})

train_ds = Dataset.from_pandas(train_df)
val_ds   = Dataset.from_pandas(val_df)
test_ds  = Dataset.from_pandas(test_df)

tokenizer = AutoTokenizer.from_pretrained("google/muril-base-cased")

def tokenize(x):
    return tokenizer(x['text'], padding='max_length', truncation=True, max_length=128)

train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)
test_ds  = test_ds.map(tokenize, batched=True)

train_ds.set_format(type='torch', columns=['input_ids','attention_mask','label'])
val_ds.set_format(type='torch', columns=['input_ids','attention_mask','label'])
test_ds.set_format(type='torch', columns=['input_ids','attention_mask','label'])

model = AutoModelForSequenceClassification.from_pretrained(
    "google/muril-base-cased", num_labels=2
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": round(accuracy_score(labels, preds), 4),
        "precision": round(precision_score(labels, preds), 4),
        "recall": round(recall_score(labels, preds), 4),
        "f1": round(f1_score(labels, preds), 4)
    }

args = TrainingArguments(
    output_dir="./muril",
    learning_rate=2e-5,
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="no",
    disable_tqdm=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

trainer.train()

print("\n=== MuRIL ===")
print("Train:", clean_eval(trainer, train_ds))
print("Val  :", clean_eval(trainer, val_ds))
print("Test :", clean_eval(trainer, test_ds))

trainer.save_model("/content/drive/MyDrive/models/muril_model")
tokenizer.save_pretrained("/content/drive/MyDrive/models/muril_model")

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

Map:   0%|          | 0/23640 [00:00<?, ? examples/s]

...

model.safetensors:   0%|          | 0.00/953M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{'eval_loss': '0.6396', 'eval_accuracy': '0.6372', 'eval_precision': '0.5766', 'eval_recall': '0.8252', 'eval_f1': '0.6788', 'eval_runtime': '23.74', 'eval_samples_per_second': '124.5', 'eval_steps_per_second': '7.792', 'epoch': '1'}
{'eval_loss': '0.5708', 'eval_accuracy': '0.7107', 'eval_precision': '0.8047', 'eval_recall': '0.4982', 'eval_f1': '0.6154', 'eval_runtime': '23.84', 'eval_samples_per_second': '123.9', 'eval_steps_per_second': '7.759', 'epoch': '2'}
{'eval_loss': '0.6166', 'eval_accuracy': '0.6975', 'eval_precision': '0.8267', 'eval_recall': '0.4414', 'eval_f1': '0.5755', 'eval_runtime': '23.64', 'eval_samples_per_second': '125', 'eval_steps_per_second': '7.827', 'epoch': '3'}
{'eval_loss': '0.5926', 'eval_accuracy': '0.7124', 'eval_precision': '0.7602', 'eval_recall': '0.5564', 'eval_f1': '0.6426', 'eval_runtime': '23.54', 'eval_samples_per_second': '125.5', 'eval_steps_per_second': '7.858', 'epoch': '4'}
{'eval_loss': '0.5996', 'eval_accuracy': '0.7059', 'eval_precision

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/models/muril_model/tokenizer_config.json',
 '/content/drive/MyDrive/models/muril_model/tokenizer.json')

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification

# ================= BERT =================

# Tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize(example):
    return tokenizer(example['text'], padding='max_length', truncation=True, max_length=128)

# Convert to datasets
train_ds = Dataset.from_pandas(train_df)
val_ds   = Dataset.from_pandas(val_df)
test_ds  = Dataset.from_pandas(test_df)

train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)
test_ds  = test_ds.map(tokenize, batched=True)

train_ds.set_format(type='torch', columns=['input_ids','attention_mask','label'])
val_ds.set_format(type='torch', columns=['input_ids','attention_mask','label'])
test_ds.set_format(type='torch', columns=['input_ids','attention_mask','label'])

# Model
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, average="weighted"),
        "recall": recall_score(labels, preds, average="weighted"),
        "f1": f1_score(labels, preds, average="weighted")
    }

# Training arguments (clean output)
training_args = TrainingArguments(
    output_dir="./bert",
    learning_rate=2e-5,
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="no",
    disable_tqdm=True,
    report_to="none"
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

# Train
trainer.train()

# Clean evaluation
def clean_eval(trainer, dataset):
    res = trainer.evaluate(dataset)
    return {
        "accuracy": round(res["eval_accuracy"], 4),
        "precision": round(res["eval_precision"], 4),
        "recall": round(res["eval_recall"], 4),
        "f1": round(res["eval_f1"], 4)
    }

print("\n=== BERT ===")
print("Train:", clean_eval(trainer, train_ds))
print("Val  :", clean_eval(trainer, val_ds))
print("Test :", clean_eval(trainer, test_ds))

# Save model
trainer.save_model("/content/drive/MyDrive/models/bert_model")
tokenizer.save_pretrained("/content/drive/MyDrive/models/bert_model")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/23640 [00:00<?, ? examples/s]

Map:   0%|          | 0/2955 [00:00<?, ? examples/s]

...

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{'eval_loss': '0.521', 'eval_accuracy': '0.7164', 'eval_precision': '0.7569', 'eval_recall': '0.7164', 'eval_f1': '0.6979', 'eval_runtime': '25.42', 'eval_samples_per_second': '116.2', 'eval_steps_per_second': '7.278', 'epoch': '1'}
{'eval_loss': '0.5958', 'eval_accuracy': '0.6788', 'eval_precision': '0.6946', 'eval_recall': '0.6788', 'eval_f1': '0.677', 'eval_runtime': '25.59', 'eval_samples_per_second': '115.5', 'eval_steps_per_second': '7.231', 'epoch': '2'}
{'eval_loss': '0.7095', 'eval_accuracy': '0.7239', 'eval_precision': '0.7525', 'eval_recall': '0.7239', 'eval_f1': '0.71', 'eval_runtime': '25.54', 'eval_samples_per_second': '115.7', 'eval_steps_per_second': '7.243', 'epoch': '3'}
{'eval_loss': '0.9042', 'eval_accuracy': '0.7195', 'eval_precision': '0.761', 'eval_recall': '0.7195', 'eval_f1': '0.701', 'eval_runtime': '25.37', 'eval_samples_per_second': '116.5', 'eval_steps_per_second': '7.291', 'epoch': '4'}
{'eval_loss': '0.9135', 'eval_accuracy': '0.7157', 'eval_precision': '

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/models/bert_model/tokenizer_config.json',
 '/content/drive/MyDrive/models/bert_model/tokenizer.json')

In [ ]:
# Load saved model
import joblib

model = joblib.load("/content/drive/MyDrive/models/logistic_model.pkl")
tfidf = joblib.load("/content/drive/MyDrive/models/tfidf_vectorizer.pkl")

# Example test
sample = ["this is hate speech example"]
sample_tfidf = tfidf.transform(sample)

print(model.predict(sample_tfidf))

[0]
